In [2]:
import tensorflow as tf
import larq as lq

In [4]:
print("Hello")

Hello


In [5]:
print("TF:", tf.__version__)
print("Larq:", larq.__version__)

TF: 2.9.0


NameError: name 'larq' is not defined

In [6]:
!pip install larq


In [7]:
print("TF:", tf.__version__)
print("Larq:", lq.__version__)

TF: 2.9.0
Larq: 0.13.3


In [22]:
import tensorflow as tf
import larq as lq
import numpy as np

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between -1 and 1
#train_images, test_images = train_images >= 127, test_images >= 127
train_images = (train_images > 127).astype(np.int32)
test_images  = (test_images  > 127).astype(np.int32)

print(test_images[120].reshape(28,28))

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 

In [25]:
import tensorflow as tf
import larq as lq
import numpy as np

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between -1 and 1
train_images, test_images = 2 *np.ceil(train_images / 255) - 1, 2 * np.ceil(test_images / 255) -1

#train_images = (train_images > 127).astype(np.int32)
#test_images  = (test_images  > 127).astype(np.int32)

# print(test_images[120].reshape(28,28))

kwargs = dict(
    input_quantizer="ste_sign",
    kernel_quantizer="ste_sign",
    kernel_constraint="weight_clip"
)

model = tf.keras.models.Sequential()

# In the first layer we only quantize the weights and not the input
model.add(lq.layers.QuantConv2D(64, (1, 1),
                                kernel_quantizer="ste_sign",
                                kernel_constraint="weight_clip",
                                use_bias=False,
                                input_shape=(28, 28, 1)))
model.add(lq.layers.QuantConv2D(32, (1, 1), use_bias=False, **kwargs))
model.add(tf.keras.layers.Flatten())
model.add(lq.layers.QuantDense(10, use_bias=False, **kwargs))
model.add(tf.keras.layers.Activation("softmax"))
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_images, train_labels, batch_size=64, epochs=6)
test_loss, test_acc = model.evaluate(test_images, test_labels)

print(f"Test accuracy {test_acc * 100:.2f} %")

Epoch 1/6
938/938 [==============================] - 30s 32ms/step - loss: 95.6944 - accuracy: 0.8209
Epoch 2/6
938/938 [==============================] - 30s 32ms/step - loss: 84.7504 - accuracy: 0.8464
Epoch 3/6
938/938 [==============================] - 29s 31ms/step - loss: 82.5068 - accuracy: 0.8519
Epoch 4/6
938/938 [==============================] - 31s 33ms/step - loss: 76.4481 - accuracy: 0.8546
Epoch 5/6
938/938 [==============================] - 36s 38ms/step - loss: 76.5919 - accuracy: 0.8549
Epoch 6/6
313/313 [==============================] - 2s 5ms/step - loss: 82.8287 - accuracy: 0.8455
Test accuracy 84.55 %


In [47]:
for row in test_images[755].reshape(28,28).astype(int):
    print(*row)

-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 1 1 1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1
-1 -1 -1 -1 -

In [48]:
model.save("mnist-minus.h5")

In [49]:
model.layers

In [50]:
model.layers[0].get_weights()
model.layers[1].get_weights()
model.layers[3].get_weights()

[array([[ 0.01056769,  0.01679685, -0.00909639, ...,  0.00578162,
          0.00087616,  0.00641698],
        [ 0.01452149,  0.00707554, -0.02123254, ...,  0.00353166,
         -0.01042676, -0.00047605],
        [ 0.00480655,  0.00897484,  0.00267445, ...,  0.00816328,
         -0.01234013, -0.0109965 ],
        ...,
        [-0.00851894,  0.0057993 , -0.00056764, ...,  0.00348351,
          0.00576437,  0.01099137],
        [ 0.00224427,  0.00545551,  0.02220064, ...,  0.00249219,
          0.01506151, -0.00193093],
        [-0.00779787,  0.01777857,  0.01286197, ...,  0.01395905,
         -0.01068689, -0.00189428]], dtype=float32)]

In [55]:
with lq.context.quantized_scope(True):
    print(model.layers[0].get_weights()[0])  # (32, 512) binarized weights
    print()
    print(model.layers[1].get_weights()[0])
    print()
    print(model.layers[3].get_weights()[0])

[[[[-1.  1. -1. -1.  1. -1. -1. -1.  1. -1.  1. -1. -1.  1.  1. -1. -1.
     1.  1. -1.  1. -1. -1. -1. -1. -1.  1. -1. -1.  1.  1. -1.  1.  1.
     1.  1.  1.  1. -1.  1. -1. -1. -1.  1. -1.  1.  1. -1.  1.  1.  1.
    -1. -1.  1.  1.  1.  1.  1.  1. -1. -1. -1.  1.  1.]]]]

[[[[-1.  1.  1. ... -1. -1.  1.]
   [ 1. -1.  1. ... -1. -1. -1.]
   [ 1. -1. -1. ...  1.  1.  1.]
   ...
   [-1.  1.  1. ... -1. -1.  1.]
   [-1. -1.  1. ... -1. -1. -1.]
   [ 1. -1. -1. ... -1.  1.  1.]]]]

[[ 1.  1. -1. ...  1.  1.  1.]
 [ 1.  1. -1. ...  1. -1. -1.]
 [ 1.  1.  1. ...  1. -1. -1.]
 ...
 [-1.  1. -1. ...  1.  1.  1.]
 [ 1.  1.  1. ...  1.  1. -1.]
 [-1.  1.  1. ...  1. -1. -1.]]


In [62]:
with lq.context.quantized_scope(True):
    print(*model.layers[0].get_weights()[0][0][0][0].astype(int))  # (32, 512) binarized weights
    print()
    for row in model.layers[1].get_weights()[0][0][0].astype(int):
        print(*row)
    print()
    for row in model.layers[3].get_weights()[0].astype(int):
        print(*row)

-1 1 -1 -1 1 -1 -1 -1 1 -1 1 -1 -1 1 1 -1 -1 1 1 -1 1 -1 -1 -1 -1 -1 1 -1 -1 1 1 -1 1 1 1 1 1 1 -1 1 -1 -1 -1 1 -1 1 1 -1 1 1 1 -1 -1 1 1 1 1 1 1 -1 -1 -1 1 1

-1 1 1 1 1 -1 -1 -1 1 -1 -1 1 -1 1 1 -1 -1 1 1 -1 -1 -1 1 -1 1 -1 1 1 1 -1 -1 1
1 -1 1 1 -1 1 1 -1 1 -1 1 -1 -1 -1 1 1 1 1 -1 -1 -1 -1 -1 1 -1 1 1 1 -1 -1 -1 -1
1 -1 -1 1 1 1 1 1 1 -1 -1 -1 1 1 -1 -1 -1 -1 1 -1 -1 1 1 1 1 -1 -1 -1 1 1 1 1
-1 1 -1 -1 1 1 1 -1 1 -1 -1 -1 1 1 1 -1 1 -1 -1 -1 1 1 1 -1 1 -1 1 1 1 -1 1 -1
1 -1 1 1 1 1 -1 1 -1 1 1 1 -1 -1 -1 -1 -1 -1 1 -1 1 -1 1 1 1 1 -1 -1 1 -1 -1 -1
-1 -1 1 -1 -1 -1 1 1 1 -1 -1 -1 1 -1 1 1 -1 1 -1 -1 -1 1 -1 -1 1 1 -1 -1 1 1 1 1
-1 -1 1 1 1 1 1 1 1 1 -1 -1 -1 -1 1 -1 -1 1 1 -1 1 1 1 -1 -1 1 -1 1 1 -1 -1 1
-1 1 1 -1 -1 1 1 1 -1 -1 1 1 1 1 -1 1 -1 -1 -1 1 1 1 -1 1 1 -1 1 1 -1 1 -1 -1
-1 1 -1 -1 1 1 -1 -1 1 1 1 -1 -1 -1 1 -1 -1 -1 1 1 1 1 -1 1 -1 1 1 1 1 1 1 1
-1 -1 1 -1 -1 1 1 1 1 1 1 1 -1 1 1 1 -1 -1 -1 1 1 1 1 1 1 1 1 -1 1 -1 -1 -1
-1 -1 1 -1 1 -1 1 1 1 -1 -1 1 -1 -1 -1 -1 1 1 1 1 -1

In [63]:
with lq.context.quantized_scope(True):
    for row in model.layers[3].get_weights()[0].astype(int):
        print(*row)

1 1 -1 -1 1 -1 -1 1 1 1
1 1 -1 -1 1 -1 1 1 -1 -1
1 1 1 -1 1 -1 -1 1 -1 -1
1 -1 1 1 1 1 1 -1 -1 1
-1 1 1 1 1 -1 1 -1 1 1
1 1 -1 -1 -1 1 -1 -1 -1 1
1 1 1 -1 1 1 -1 1 -1 1
-1 -1 1 -1 -1 1 1 -1 -1 1
-1 -1 1 1 -1 -1 -1 1 1 1
-1 -1 1 1 1 1 1 1 -1 1
-1 1 -1 -1 1 -1 -1 1 -1 1
-1 -1 1 1 -1 1 1 -1 -1 1
-1 1 -1 -1 -1 -1 1 -1 1 1
-1 -1 1 -1 -1 1 1 -1 1 -1
1 -1 -1 -1 1 -1 -1 1 1 1
-1 -1 -1 -1 -1 -1 1 1 1 1
-1 1 1 -1 1 -1 -1 -1 -1 1
1 1 -1 1 -1 1 -1 -1 1 -1
-1 -1 1 -1 -1 1 1 1 1 -1
-1 1 1 -1 1 -1 -1 1 -1 -1
-1 -1 1 -1 -1 -1 1 1 -1 1
-1 1 -1 -1 1 -1 1 -1 1 1
1 -1 1 1 -1 1 1 1 1 1
-1 1 -1 -1 1 1 -1 -1 1 1
1 -1 1 1 1 -1 1 -1 1 -1
-1 1 -1 -1 -1 -1 -1 1 -1 1
1 -1 1 1 -1 -1 1 -1 1 1
1 -1 1 1 1 1 1 1 -1 1
-1 1 1 1 -1 1 1 1 -1 -1
1 -1 -1 1 1 -1 -1 -1 -1 -1
1 -1 -1 1 -1 -1 1 -1 1 -1
1 1 -1 -1 -1 -1 -1 1 1 1
-1 1 -1 1 1 -1 -1 -1 1 -1
-1 1 1 -1 -1 1 1 -1 1 1
1 1 -1 1 1 1 1 -1 -1 1
-1 1 1 -1 -1 -1 -1 1 -1 -1
1 -1 -1 -1 1 -1 1 1 -1 -1
-1 -1 1 -1 1 1 -1 -1 -1 -1
1 1 1 1 -1 1 1 -1 1 -1
-1 1 1 -1 1 1 1 -1 -1 -1
-1 